// ID (hex), startBit, endBit, resolution, offset (aplied BEFORE resolution multiplication), decimals, unit, 
//     requestID (hex string), responseID (hex string), options (hex, see MainActivity for definitions), optional name, optional list

                        + "7ec,24,39,0.5,0,2,V,223203,623203,ff\n" // HV Battery voltage
                        + "7ec,24,39,0.25,32768,2,A,223204,623204,ff\n" // HV Battery current

                        + "7ec,24,31,1,0,0,%,223206,623206,ff\n" // Battery health in %
                        + "658,33,39,1,0,0,%,,,ff\n" // Battery Health

                        + "7ec,24,39,2,0,2,%,222002,622002,e2\n" // SOC
                        + "7ec,24,39,2.083333333,0,2,%,222002,622002,e5\n" // SOC
                        + "7bb,192,207,0.01,0,2,%,2103,6103,e2\n" // Real State of Charge
                        + "654,25,31,1,0,0,,,,ff\n" // State of Charge

                        + "7bb,336,351,0.01,0,2,kW,2101,6101,e2\n" // Maximum battery input power
                        + "427,40,47,0.3,0,0,kW,,,e2\n" // Available Charging Power


In [7]:
import can
import isotp
import udsoncan
import udsoncan.configs
from udsoncan import AsciiCodec, Request
from udsoncan.services import ReadDataByIdentifier
from udsoncan.client import Client
#from udsoncan.connections import IsoTPSocketConnection
from udsoncan.connections import PythonIsoTpConnection
from udsoncan.exceptions import (
    InvalidResponseException,
    NegativeResponseException,
    TimeoutException,
    UnexpectedResponseException,
)

class UInt16Codec(udsoncan.DidCodec):
    def encode(self, val: int) -> bytes:
        return int.to_bytes(val)

    def decode(self, payload: bytes) -> int:
        return int.from_bytes(payload)

    def __len__(self):
        #raise udsoncan.DidCodec.ReadAllRemainingData  # Dynamically handles remaining payload length
        return 2

class Int16Codec(udsoncan.DidCodec):
    def encode(self, val: int) -> bytes:
        return int.to_bytes(val, signed=True)

    def decode(self, payload: bytes) -> int:
        return int.from_bytes(payload, signed=True)

    def __len__(self):
        #raise udsoncan.DidCodec.ReadAllRemainingData  # Dynamically handles remaining payload length
        return 2

class Fixed16Codec(udsoncan.DidCodec):
    def __init__(self, scale: float, offset: int = 0):
        self.scale = scale
        self.offset = offset

    def encode(self, val) -> bytes:
        return int.to_bytes(self.offset + int(val / self.scale))

    def decode(self, payload) -> float:
        return self.scale * (int.from_bytes(payload) - self.offset)

    def __len__(self):
        return 2


def make_uds_request():
    # HV Voltage
    did_v_hv = 0x3203
    # Battery Current
    did_i_bat = 0x3204
    # SOC
    did_soc = 0x2002

    dids_requested = [did_v_hv, did_i_bat, did_soc]

    tp_address = isotp.Address(
        isotp.AddressingMode.Normal_11bits,
        #isotp.AddressingMode.Extended_29bits,
        # txid=0x7DF,7E0,7E4 and rxid=0x7E8,7EC are good candidates
        txid=0x7E4,
        rxid=0x7EC,
    )

    config = udsoncan.configs.default_client_config.copy()
    config["data_identifiers"] = {
        did_v_hv: Fixed16Codec(0.5),
        did_i_bat: Fixed16Codec(0.25, 32768),
        did_soc: Fixed16Codec(0.02),
    }

    # Create udsoncan connection layer
    bus = can.Bus("can_spi", interface="socketcan")
    notifier = can.Notifier(bus, [])
    # Refer to isotp documentation for full details about parameters.
    tp_params = {
        # Link layer (CAN layer) works with 8 byte payload (CAN 2.0)
        "tx_data_length": 8,
        # Minimum length of CAN messages. When different from None, messages are
        # padded to meet this length. Works with CAN 2.0 and CAN FD.
        "tx_data_min_length": 8,
        # Will pad all transmitted CAN messages with byte 0x00.
        "tx_padding": 0x00,
    }
    stack = isotp.NotifierBasedCanStack(
        bus,
        notifier=notifier,
        address=tp_address,
        params=tp_params,
    )
    connection = PythonIsoTpConnection(stack)

    with Client(connection, config=config, request_timeout=2.0) as client:        
        try:
            print("Sending request to read Data Identifier: ...")

            # Read Data By Identifier (Service 0x22).
            response = client.read_data_by_identifier(dids_requested)

            # Print interpreted response data.
            print(
                f"V_HV: {response.service_data.values[did_v_hv]} V\n",
                f"I_BAT: {response.service_data.values[did_i_bat]} A\n",
                f"SOC: {response.service_data.values[did_soc]} %\n"
            )
            
        except NegativeResponseException as e:
            print(f"ECU rejected the request with code: {e.response.code_name} (0x{e.response.code:02X})")
        except InvalidResponseException as e:
            print(f"Received an invalid or malformed response: {e}")
        except Exception as e:
            print(f"An unexpected error occurred: {e}")

if __name__ == "__main__":
    make_uds_request()

Sending request to read Data Identifier: ...
V_HV: 376.5 V
 I_BAT: -1.0 A
 SOC: 46.24 %

V_HV: 376.5 V
 I_BAT: -0.75 A
 SOC: 46.24 %



In [15]:
"\x54

'^'

In [18]:
0x54

84